In [13]:
# papermill parameters

garch_lookback=1000
garch_dist='skewt'
input_file='usdt.parquet'
data_output_file='binance-1d-spot-volatility-forecast.parquet'
eval_output_file='binance-1d-spot-volatility-eval.csv'
jobs_concurrency=2
jobs_backend='threading'

In [14]:
print(f'''
GJR-GARCH volatility forecast evaluation
========================================

input_file={input_file}
data_output_file={data_output_file}
eval_output_file={eval_output_file}

garch_lookback={garch_lookback}
garch_dist={garch_dist}

jobs_concurrency={jobs_concurrency}
jobs_backend={jobs_backend}
''')


GJR-GARCH volatility forecast evaluation

input_file=usdt.parquet
data_output_file=binance-1d-spot-volatility-forecast.parquet
eval_output_file=binance-1d-spot-volatility-eval.csv

garch_lookback=1000
garch_dist=skewt

jobs_concurrency=2
jobs_backend=threading



Fit GJR-GARCH model and forecast
================================

In [16]:
import polars as pl
import duckdb
from arch import arch_model
import numpy as np
from tqdm import tqdm

con = duckdb.connect()

con.execute(f'''
    create or replace view returns as
    select 
        ts, 
        symbol,
        ln(close / lag(close) over (partition by symbol order by ts)) as ret
    from read_parquet('{input_file}')
    where open > 0 and high > 0 and low > 0 and close > 0 and base_volume > 0
''')

limits = con.execute('''
    select
        min(ts) as min,
        max(ts) as max,
        symbol,
        count(ts) as rows
    from returns
    group by symbol
    having rows >= ?
''', [garch_lookback]).pl()
limits

min,max,symbol,rows
"datetime[μs, Asia/Ho_Chi_Minh]","datetime[μs, Asia/Ho_Chi_Minh]",str,i64
2017-08-17 07:00:00 +07,2026-02-19 07:00:00 +07,"""ETHUSDT""",3109
2020-10-15 07:00:00 +07,2026-02-19 07:00:00 +07,"""FILUSDT""",1954
2021-04-23 07:00:00 +07,2026-02-19 07:00:00 +07,"""FORTHUSDT""",1764
2021-08-30 07:00:00 +07,2026-02-19 07:00:00 +07,"""GNOUSDT""",1635
2023-02-17 07:00:00 +07,2026-02-19 07:00:00 +07,"""GNSUSDT""",1099
…,…,…,…
2021-11-12 07:00:00 +07,2024-11-06 07:00:00 +07,"""KP3RUSDT""",1091
2021-02-04 07:00:00 +07,2025-02-10 07:00:00 +07,"""LITUSDT""",1468
2022-10-28 07:00:00 +07,2026-02-19 07:00:00 +07,"""OSMOUSDT""",1211


In [17]:
from joblib import Parallel, delayed


def process(df: pl.DataFrame) -> pl.DataFrame:
    def forecast_sigma(s: pl.Series) -> dict:
        try:
            gjr = arch_model(
                s.to_numpy() * 100,
                vol='Garch',
                p=1,
                o=1,
                q=1,
                dist=garch_dist,
                rescale=False
            ).fit(
                disp='off',
            )
            return {
                'sigma_gjr': np.sqrt(gjr.forecast(horizon=1).variance.values[-1, 0]) / 100,
                'gjr_mu': gjr.params.get('mu', np.nan),
                'gjr_omega': gjr.params.get('omega', np.nan),
                'gjr_alpha': gjr.params.get('alpha[1]', np.nan),
                'gjr_gamma': gjr.params.get('gamma[1]', np.nan),
                'gjr_beta': gjr.params.get('beta[1]', np.nan)
            }
        except Exception as e:
            print(f'''caught {e}, returning NaN''')
            return {
                'sigma_gjr': np.nan,
                'gjr_mu': np.nan,
                'gjr_omega': np.nan,
                'gjr_alpha': np.nan,
                'gjr_gamma': np.nan,
                'gjr_beta': np.nan
            }
            
    return df.select([
        pl.col("ret")
            .rolling_map(forecast_sigma, window_size=garch_lookback)
            .alias('gjr'),
        pl.col('ts'),
        pl.col('symbol'),
    ]).unnest('gjr').with_columns([
        pl.col('sigma_gjr').shift(1),
    ])
    
syms = df.partition_by('symbol')
fc = Parallel(n_jobs=jobs_concurrency, backend=jobs_backend)(
    delayed(process)(s) for s in tqdm(syms, desc="forecasting volatility")
)

pl.concat(fc).write_parquet(data_output_file)

NameError: name 'df' is not defined

Evaluate model against daily R/S volatility
===========================================

In [ ]:
res = (
    df.join(pl.read_parquet(data_output_file), on=['ts','symbol'])
        .drop_nulls()
        .sort(['symbol','ts'])
        .group_by('symbol')
        .agg([
            ((pl.col("sigma_gjr") - pl.col("sigma_rs"))**2).mean().sqrt().alias("rmse"),
            (pl.col("sigma_gjr") - pl.col("sigma_rs")).abs().mean().alias("mae"),    
            (pl.col("sigma_gjr") - pl.col("sigma_rs")).mean().alias("bias")
        ])
)
res.select([pl.col('rmse'), pl.col('mae'), pl.col('bias')]).describe().write_csv(eval_output_file)

In [ ]:
import matplotlib.pyplot as plt

# Metrics to plot
metrics = ["rmse", "mae", "bias"]
colors = ["#3498db", "#e74c3c", "#2ecc71"]

plt.figure(figsize=(18, 6))

# Create a layout with 3 subplots
for i, metric in enumerate(metrics):
    plt.subplot(1, 3, i + 1)
    
    # We use .to_numpy() to ensure compatibility with matplotlib
    plt.hist(res[metric].to_numpy(), bins=20, color=colors[i], edgecolor='black', alpha=0.7)
    
    plt.title(f'Distribution of {metric.upper()}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()